# MLP.ipynb（详细中文注释版）

Karpathy《building makemore Part 2: MLP》跟练笔记本。相比第 1 讲的 bigram（只看前 1 个字符），
这里用 **MLP 多层感知机** + **前 3 个字符**的上下文来预测下一个字符，并引入**词嵌入(embedding)**。

本文件在你原始 notebook 上**逐行加中文注释**（重点解释每行含义 + 括号里各个值/参数的意思），**代码逻辑一行未改**。
原始版本已备份为 `MLP.BACKUP.ipynb`。

> ⚠️ 我在你代码里发现了 4 处**原有问题**（不是我改出来的），已在对应行用注释标出，方便你按需修正：
> 1. **cell 0**：`from sympy.printing.pytorch import torch` 会把真正的 torch 覆盖掉，导致后面全崩 → 已注释掉。
> 2. **cell 7**：`n1=0.9`、`n2=0.8` 写反了，导致**验证集为空**（`words[n1:n2]` 取不到东西）。
> 3. **cell 31**：`loss.backward` 少了括号 `()`，其实**没有真正执行反向传播**。
> 4. **cell 38/40**：注释里的形状 `(32,3,2)`/`(32,100)` 是旧的；现在嵌入是 10 维、隐藏层 200，实际形状不同。

## 1. 导入 + 读数据

In [ ]:
import torch                       # PyTorch 主库(张量运算 + 自动求导)
import torch.nn.functional as F    # 函数式接口:one_hot / cross_entropy / softmax 等
import  random                     # Python 随机库,用来打乱数据
import  matplotlib.pyplot as plt   # 画图库
# from sympy.printing.pytorch import torch   # ⚠️ 这行是 IDE 误加的!会把上面的 torch 覆盖成 sympy 的对象,导致后面全部报错,已注释(建议删除)
%matplotlib inline                 # 让图像内嵌在 notebook 里显示

In [ ]:
words=open('names.txt','r').read().splitlines()  # 读入 names.txt,按换行切成列表(每个元素是一个名字)
words[:8]                                          # 看前 8 个名字([:8]=取前 8 个元素)

In [ ]:
len(words)   # 名字总数(约 32033 个)

In [ ]:
chars=sorted(list(set(''.join(words))))    # ''.join 把所有名字拼成一个大字符串; set 去重; list 转列表; sorted 排序 -> ['a'..'z']
stoi={s:i+1 for i ,s in enumerate(chars)}  # 字符->编号; enumerate 从 0 开始, +1 让字母占 1~26
stoi['.']=0                                 # '.' 作为开始/结束标记, 编号 0
itos={i:s for s,i in stoi.items()}          # 编号->字符; 把 stoi 的键值对反过来
print(itos)                                 # 打印「编号->字符」映射表

## 2. 构造训练数据（滑动窗口）

`block_size=3` 表示用**前 3 个字符**预测第 4 个。用滑动窗口把每个名字切成很多组 (上下文 → 目标)。
下面的 `print(... '--->' ...)` 会把每一条样本打印出来，方便你直观看到数据长什么样（数据量大时可删掉这行）。

In [ ]:
block_size=3                               # 上下文长度:用前 3 个字符预测下一个字符
X,Y=[],[]                                   # X=输入(每条是3个字符编号), Y=目标(下一个字符编号)
for w in words:                             # 遍历每个名字
    context=[0]*block_size                  # 初始上下文=[0,0,0](即3个'.')
    for ch in w+'.':                        # 遍历名字每个字符,末尾补 '.' 作为结束目标
        ix=stoi[ch]                         # 当前字符编号
        X.append(context)                   # 记录当前上下文(3个编号)
        Y.append(ix)                        # 记录目标字符
        print(''.join(itos[i] for i in context), '--->', itos[ix])  # 打印「上下文 ---> 目标」(直观看数据)
        context=context[1:]+[ix]            # 滑动窗口:去掉最旧的[1:],把当前字符接到末尾
X=torch.tensor(X)                           # 列表转张量, 形状 (样本数, 3)
Y=torch.tensor(Y)                           # 列表转张量, 形状 (样本数,)

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype   # 查看:X是(样本数,3)整数, Y是(样本数,)整数

把构造数据的逻辑封装成函数，方便对训练/验证/测试集分别调用：

In [ ]:
def build_dataset(words):                  # 封装:输入名字列表 -> 输出 (X, Y)
    block_size=3                            # 上下文长度 3
    X,Y=[],[]                               # 输入 / 目标
    for w in words:                         # 遍历名字
        context=[0]*block_size              # 初始上下文 3 个 '.'
        for ch in w+'.':                    # 遍历字符 + 结束符
            ix=stoi[ch]                     # 字符编号
            X.append(context)               # 记录上下文
            Y.append(ix)                    # 记录目标
            print(''.join(itos[i] for i in context), '--->', itos[ix])  # 打印数据(数据量大时建议删掉,否则刷屏且文件超大)
            context=context[1:]+[ix]        # 滑动窗口
    X=torch.tensor(X)                       # 转张量
    Y=torch.tensor(Y)                       # 转张量
    return X,Y                              # 返回输入和目标

In [ ]:
random.seed(42)                            # 固定随机种子(42 只是惯用数字),保证每次打乱结果一致
random.shuffle(words)                      # 原地打乱名字顺序
n1=int(0.9*len(words))                      # 90% 处的下标
n2=int(0.8*len(words))                      # 80% 处的下标
# ⚠️ 这里 n1(0.9) > n2(0.8) 写反了! 导致下面 words[n1:n2]=words[0.9:0.8] 是空的 -> 验证集为空
# 正确写法应为:  n1=int(0.8*len(words));  n2=int(0.9*len(words))
Xtr,Ytr=build_dataset(words[:n1])           # 训练集: 取前 n1 个名字
Xdev,Ydev=build_dataset(words[n1:n2])       # 验证集: n1~n2 (受上面 bug 影响,当前为空)
Xte,Yte=build_dataset(words[n2:])           # 测试集: n2 之后
print(Xdev,Ydev)                            # 打印验证集(当前为空)

## 3. 词嵌入（embedding）

`C` 是**嵌入表**：把每个字符编号映射成一个低维向量。用 `C[X]` 一次性把所有输入编号批量换成向量。
`C[5]` 等价于「5 号字符的 one-hot 向量 × C」，但直接索引更快。

In [ ]:
C=torch.randn(27,2)   # 嵌入表: 27=词表大小(字符数), 2=每个字符嵌入成 2 维向量; randn=标准正态随机初始化

In [ ]:
C[5]   # 取 5 号字符的 2 维嵌入向量

In [ ]:
F.one_hot(torch.tensor(5),num_classes=27).float() @ C   # 验证: 5 的 one-hot(长度 num_classes=27)乘嵌入表 = C[5]; .float()转浮点才能做矩阵乘 @

In [ ]:
C[X].shape   # 用整个 X 去索引嵌入表; X 是(样本数,3) -> 结果 (样本数, 3, 2): 每个字符都取到 2 维向量

In [ ]:
emb=C[X]        # 把输入的字符编号批量换成嵌入向量
emb.shape       # (样本数, 3, 2): 3 个上下文字符, 每个 2 维

## 4. 隐藏层（手动搭一层 MLP）

把 3 个字符的嵌入**拼接**成一个长向量(3×2=6 维)，过一个全连接层 + `tanh` 激活。
`view(-1, 6)` 的 `-1` 表示这一维让 PyTorch 自动算(=样本数)，`6` 是拼接后的维度。

In [ ]:
w1=torch.randn((6,100))   # 第一层权重: 6=输入维(3字符×2维), 100=隐藏层神经元个数
b1=torch.randn(100)        # 第一层偏置: 长度 100, 对应 100 个神经元

In [ ]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape   # 把3个字符嵌入拼接; emb[:,0,:]=所有样本第1个字符的嵌入; 末尾的 1=沿第1维(列)拼接 -> (样本数,6)

In [ ]:
torch.cat(torch.unbind(emb,1),1)   # 更通用写法: unbind(emb,1)沿第1维拆成3块, cat(...,1)再沿第1维拼回, 效果同上

In [ ]:
h=torch.tanh(emb.view(-1,6)@w1+b1)   # emb.view(-1,6)展平成(样本数,6)(-1=自动算样本数); @w1矩阵乘; +b1加偏置; tanh激活 -> 隐藏层输出
h                                     # 查看 h

In [ ]:
h.shape   # (样本数, 100)

In [ ]:
(emb.view(-1,6)@w1).shape   # 验证矩阵乘的形状: (样本数, 100)

## 5. 输出层 + 手动 softmax + loss

隐藏层再过一层得到 27 个字符的得分(logits)，`exp`→归一化得到概率，再取正确目标的负对数似然当 loss。

In [ ]:
W2 = torch.randn((100, 27))   # 第二层权重: 100=隐藏层维, 27=输出维(词表大小)
b2 = torch.randn(27)           # 第二层偏置: 长度 27

In [ ]:
logits = h @ W2 + b2   # 输出层: 隐藏层 h 乘 W2 再加 b2, 得到每个字符的原始得分
logits                  # 查看

In [ ]:
logits.shape   # (样本数, 27)

In [ ]:
counts=logits.exp()   # 得分取指数 -> 全为正数(相当于'计数')

In [ ]:
prob=counts/counts.sum(1,keepdim=True)   # 按行归一化成概率; sum(1,...)沿第1维(每行)求和; keepdim=True保持(样本数,1)以便广播相除

In [ ]:
prob.shape   # (样本数, 27)

In [ ]:
loss=-prob[torch.arange(X.shape[0]),Y].log().mean()   # 取每个样本'正确目标'的概率; arange(X.shape[0])=行下标0..N-1, Y=列下标(目标); log取对数, 取负, mean求平均 = loss

In [ ]:
loss   # 查看当前 loss

## 6. 正式版：更大的嵌入 + 参数汇总

把嵌入维度从 2 提到 10、隐藏层从 100 提到 200，表达力更强。用固定种子 `g` 保证可复现。

In [ ]:
Xtr.shape, Ytr.shape   # 训练集形状

In [ ]:
g = torch.Generator().manual_seed(2147483647)      # 随机数发生器, 固定种子(2147483647是常用大质数)保证可复现
C=torch.randn((27,10),generator=g)                  # 嵌入表: 27字符, 每个10维(比前面2维表达力更强)
W1 = torch.randn((30, 200), generator=g)            # 第一层: 30=3字符×10维, 200=隐藏神经元数
b1 = torch.randn(200, generator=g)                  # 第一层偏置
W2 = torch.randn((200, 27), generator=g)            # 第二层: 200 -> 27
b2 = torch.randn(27, generator=g)                   # 第二层偏置
parametes=[C, W1, b1, W2, b2]                        # 收集所有参数(注意变量名拼成了 parametes,少个 r,后面一直沿用)
parametes                                            # 查看参数列表

In [ ]:
sum(p.nelement() for p in parametes)   # 所有参数的元素总数(nelement=元素个数),约 11897

In [ ]:
F.cross_entropy(logits,Y)   # 交叉熵损失: 内部=softmax+负对数似然,比手写数值更稳; 参数(预测得分, 真实目标)

In [ ]:
for p in parametes:        # 遍历所有参数
  p.requires_grad = True   # 开启梯度追踪(训练前必须设置)
loss.backward              # ⚠️ 这里漏了括号! 只是'引用'了函数并没有调用; 要真正反向传播应写成 loss.backward()

## 7. 训练

`lre`/`lrs` 是 Karpathy 用来**探索合适学习率**的技巧：在 10⁻³~10⁰ 之间取值试跑，看哪个 loss 降得好。
正式训练里改用了固定的分段学习率(0.1 → 0.01)。

In [ ]:
lre = torch.linspace(-3, 0, 1000)   # 在[-3,0]之间均匀取1000个数; (起点-3, 终点0, 个数1000)
lrs = 10**lre                        # 取10的幂 -> 学习率范围 10^-3(=0.001) 到 10^0(=1)

In [ ]:
lri = []     # 记录尝试过的学习率
lossi = []   # 记录对应的 loss
stepi = []   # 记录步数

In [ ]:
for i in range(200000):                          # 训练 20 万步

  # minibatch construct                          # 构造小批量(mini-batch)
  ix = torch.randint(0, Xtr.shape[0], (32,))     # 随机抽32个样本下标; (0=最小值, Xtr.shape[0]=最大值即样本数, (32,)=要32个)

  # forward pass                                 # 前向传播
  emb = C[Xtr[ix]] # (32, 3, 10)                 # 取这批的嵌入; 形状(32批, 3字符, 10维)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)   # 展平成(32,30)过第一层+tanh -> (32,200)
  logits = h @ W2 + b2 # (32, 27)                # 输出层 -> (32,27)
  loss = F.cross_entropy(logits, Ytr[ix])        # 交叉熵 loss(这批的目标 Ytr[ix])
  #print(loss.item())                            # (调试用:打印当前loss)

  # backward pass                                # 反向传播
  for p in parametes:                            # 遍历参数
    p.grad = None                                # 清空上一步梯度
  loss.backward()                                # 求梯度(这里有括号,正确)

  # update                                       # 更新参数
  #lr = lrs[i]                                    # (旧写法:用上面探索的学习率)
  lr = 0.1 if i < 100000 else 0.01               # 前10万步 lr=0.1, 之后 0.01(学习率衰减)
  for p in parametes:                            # 遍历参数
    p.data += -lr * p.grad                       # 沿梯度反方向更新

  # track stats                                  # 记录统计信息
  #lri.append(lre[i])                             # (旧:记录学习率)
  stepi.append(i)                                # 记录步数
  lossi.append(loss.log10().item())              # 记录 loss 的 log10(画图更平滑)

下面这段和上面几乎一样（又训练 20 万步）。功能重复，实际保留一个即可：

In [ ]:
for i in range(200000):                          # 又训练 20 万步(与上一段功能重复)
    ix =torch.randint(0,Xtr.shape[0],(32,))      # 随机抽32个样本下标

    emb=C[Xtr[ix]]                               # 取嵌入 (32,3,10)
    h=torch.tanh(emb.view(-1,30) @ W1 + b1)      # 隐藏层 (32,200)
    logits=h@W2 +b2                              # 输出层 (32,27)
    loss=F.cross_entropy(logits,Ytr[ix])         # 交叉熵 loss
    for p in parametes:                          # 遍历参数
        p.grad=None                              # 清空梯度
    loss.backward()                              # 反向传播
    lr = 0.1 if i < 100000 else 0.01             # 分段学习率
    for p in parametes:                          # 遍历参数
        p.data+=-lr*p.grad                       # 梯度下降更新
    #lri.append(lre[i])                           # (旧:记录学习率)
    stepi.append(i)                              # 记录步数
    lossi.append(loss.log10().item())            # 记录 log10(loss)

In [ ]:
logits.max(1)   # 沿第1维取每行最大值; 返回(每行最大值, 对应的列索引即预测字符)

In [ ]:
plt.plot(stepi, lossi)   # 画 loss(log10)随步数变化的曲线, 总体应下降

## 8. 评估：训练集 / 验证集 loss

> 注：代码里 `# (32, 3, 2)` 等是旧注释；现在嵌入是 10 维、隐藏层 200，实际形状是 (样本数,3,10) 和 (样本数,200)。

In [ ]:
emb = C[Xtr] # (32, 3, 2)                        # 整个训练集的嵌入(注释是旧的,实际 (样本数,3,10))
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)   # 隐藏层(实际 (样本数,200))
logits = h @ W2 + b2 # (32, 27)                  # 输出层 (样本数,27)
loss = F.cross_entropy(logits, Ytr)              # 训练集整体 loss
loss                                              # 查看

In [ ]:
emb = C[Xdev] # (32, 3, 2)                        # 验证集嵌入(注意:cell7 的 bug 使 Xdev 为空,这里会报错/无意义)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)   # 隐藏层
logits = h @ W2 + b2 # (32, 27)                  # 输出层
loss = F.cross_entropy(logits, Ydev)             # 验证集 loss
loss                                              # 查看

## 9. 可视化嵌入 + 生成名字

散点图把每个字符的嵌入画出来（只有嵌入是 2 维时才直观；现在是 10 维，这里只取了前 2 维展示）。
最后从模型采样生成新名字。

In [ ]:
plt.figure(figsize=(8,8))                        # 8x8 画布
plt.scatter(C[:,0].data, C[:,1].data, s=200)     # 画嵌入散点; C[:,0]取第0维, C[:,1]取第1维; s=200=点的大小(嵌入10维时仅展示前两维)
for i in range(C.shape[0]):                       # 遍历每个字符(C.shape[0]=27)
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')  # 在每个点位置标上字符
plt.grid('minor')                                 # 显示网格

In [ ]:
context = [0] * block_size                        # 初始上下文 [0,0,0]
C[torch.tensor([context])].shape                  # 查看形状 (1,3,10): 1个样本, 3字符, 每个10维

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)   # 固定种子(+10 换一批不同结果)

for _ in range(20):                                   # 生成 20 个名字

    out = []                                          # 存字符编号
    context = [0] * block_size # initialize with all ...  # 初始上下文全是 '.'
    while True:                                        # 循环采样直到结束
      emb = C[torch.tensor([context])] # (1,block_size,d)  # 当前上下文嵌入 (1,3,10)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)       # 隐藏层; view(1,-1)展平成(1,30)
      logits = h @ W2 + b2                            # 输出得分
      probs = F.softmax(logits, dim=1)                # softmax 转概率; dim=1沿字符维归一化
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()  # 按概率抽1个字符(num_samples=1=抽1个)
      context = context[1:] + [ix]                    # 滑动窗口更新上下文
      out.append(ix)                                  # 记录字符
      if ix == 0:                                     # 抽到 '.'(编号0)结束
        break                                       # 跳出循环,结束当前名字
    print(''.join(itos[i] for i in out))              # 把编号翻译回字母并打印